# Ollama 설치 + 로컬 LLM 비교 + Modelfile + LangChain 연동

1. Ollama 설치 및 Llama3 / Mistral / Gemma 로컬 실행
2. 동일 제조 질문에 대한 세 모델 응답 비교
3. Modelfile 작성으로 "엔진부품 설비 진단 전문가" 커스텀 모델 등록



## 0. Ollama 설치

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip install -q --upgrade \
  langchain langchain-core langchain-community langchain-classic \
  langchain-huggingface langchain-ollama \
  chromadb sentence-transformers \
  opentelemetry-api opentelemetry-sdk \
  opentelemetry-exporter-otlp-proto-common opentelemetry-exporter-otlp-proto-grpc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 116.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


- 프로세스 재시작

In [4]:
!pkill -9 ollama

In [5]:
import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 1706 )


In [6]:
!ollama --version

ollama version is 0.30.10


- 프로세스 확인

In [7]:
import time
time.sleep(2)
if process.poll() is not None:
    out, err = process.communicate()
    print("프로세스 종료됨! STDOUT:", out)
    print("STDERR:", err)
else:
    print("프로세스 (PID:", process.pid, ")")


프로세스 (PID: 1706 )


- 모형 로딩-최초1회

In [8]:
# 최초 1회 워밍업 - 타임아웃을 넉넉하게(120초) 줘서 로딩이 끝까지 끝나게 둠
!curl -s --max-time 120 http://localhost:11434/api/generate \
  -d '{"model": "engine-diagnosis-expert", "prompt": "안녕", "stream": false, "options": {"num_predict": 5}}'

{"error":"model 'engine-diagnosis-expert' not found"}

## 1. 모델 다운로드: Llama3 / Mistral / Gemma

모델 크기가 크므로(4~5GB 내외) 다운로드에 시간 소요: 크기 작은 양자화 태그 모형 사용


In [9]:

#!ollama pull mistral:7b-instruct-q4_0
!ollama pull mistral:7b-instruct-q2_K
#!ollama pull gemma:7b-instruct-q4_0
!ollama pull gemma:2b
#!ollama pull llama3:8b-instruct-q4_0
!ollama pull llama3.2:3b
!ollama pull qwen2.5:3b

In [10]:
!ollama list

NAME                        ID              SIZE      MODIFIED               
qwen2.5:3b                  357c53fb659c    1.9 GB    Less than a second ago    
llama3.2:3b                 a80c4f17acd5    2.0 GB    17 seconds ago            
gemma:2b                    b50d6c999e59    1.7 GB    38 seconds ago            
mistral:7b-instruct-q2_K    1344ecf13c2e    3.1 GB    55 seconds ago            


In [11]:
!free -h
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.1Gi        56Gi       2.0Mi        25Gi        81Gi
Swap:             0B          0B          0B
Thu Jun 25 11:51:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0        

## 2. 동일 질문으로 세 모델 응답 비교


In [12]:
import requests
import json as json_lib

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(model, prompt, temperature=0.3, keep_alive="0s", timeout=120):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature, "num_predict":200,}, #생성 토큰 수 제한
            "keep_alive": "10m",  # 응답 후 즉시 메모리에서 내림            
        },
        timeout=timeout,
    )
    return response.json()["response"]

#models_to_compare = ["llama3:8b-instruct-q4_0", "mistral:7b-instruct-q4_0", "gemma:7b-instruct-q4_0"]
#models_to_compare = ["llama3.2:3b", "mistral:7b-instruct-q2_K", "gemma:2b"]
models_to_compare = ["llama3.2:3b", "gemma:2b", "qwen2.5:3b"] 


In [13]:
question = "크랭크샤프트 표면에서 크랙이 발생하는 주요 원인 3가지를 알려주세요."

for model_name in models_to_compare:
    print(f"\n{'='*20} {model_name} {'='*20}")
    start = time.time()
    answer = ask_ollama(model_name, question)
    elapsed = time.time() - start
    print(f"(응답 시간: {elapsed:.1f}초)")
    print(answer)


==================== llama3.2:3b ====================
(응답 시간: 83.8초)
크랭크샤프트 표면에서 크랙이 발생하는 주요 원인은 다음과 같습니다.

1. **thermal shock**: 열적 충격은 샤프트의 표면에 크랙을 일으킬 수 있습니다. 샤프트가 높은 온도에서 낮은 온도로 전환되는 경우, 샤프트의 내부와 외부의 온도가 다르기 때문에 열적 충격이 발생할 수 있습니다.

2. **mechanical stress**:机械적인 부담은 샤프트의 표면에 크랙을 일으킬 수 있습니다. 샤프트가 강력한 힘을 받거나, 부서진 부분이 있는 경우, 샤프트의 표면에 크랙이 발생할 수 있습니다.

3. **corrosion**: 산화 및 유기화와 같은 화학적 반응은 샤프트의 표면에 크랙을 일으킬 수 있습니다. 샤프트가 특정 환경에서 노출

==================== gemma:2b ====================
(응답 시간: 66.5초)
1. **크랙 형성 원인:** 크랙이 발생하는 주요 원인은 크랙 형성 과정에서 발생하는 여러 요인이 복합적으로 작용하는 데 기여합니다. 크랙 형성 과정에서 가장 중요한 요인은 **열과 기계적 하중**입니다.
2. **크랙 형성 환경:** 크랙이 발생하는 환경은 크랙 형성 과정에 영향을 미칠 수 있습니다. 특히, **습도, 온도, 기압, 그리고 마그네심**은 크랙 형성에 영향을 미칠 수 있습니다.
3. **크랙 형성 물리적 특성:** 크랙이 발생하는 물리적 특성 또한 크랙 형성 과정에 영향을 미칠 수 있습니다. 특히, **크랙의 크기, 형상, 그리고 색깔**은 크랙 형성 과정

==================== qwen2.5:3b ====================
(응답 시간: 13.0초)
크랭크샤프트의 표면에 발생하는 크랙은 여러 가지 이유로 인해 발생할 수 있습니다. 다음은 주요 원인 중 세 가지입니다:

1. 열적 스트레스: 엔진이 작동하면서 발생하는 열과 충격으로 인해

- 응답 속도: 모델 크기/양자화 수준에 따라 차이
- 응답 스타일: Llama3는 비교적 구조화된 답변, Mistral은 간결, Gemma는 안전성 중시 경향
- 도메인 지식 부재: 세 모델 모두 일반 상식 수준으로 답변


## 3. Modelfile 작성: 엔진부품 설비 진단 전문가 커스텀 모델

Modelfile로 시스템 프롬프트를 고정해, 매번 프롬프트에 역할 설명을 반복하지 않아도 되는 전용 모델


In [14]:
#응답시간 오래걸릴 경우 모형 변경:  llama3:8b-instruct-q4_0

modelfile_content = '''
FROM llama3.2:3b

SYSTEM """
당신은 자동차 엔진부품 제조 공장의 설비 진단 전문가입니다.
다음 원칙을 반드시 지켜 답변하세요.

1. 제공된 정보(context)에 근거해서만 답변하고, 모르면 "제공된 정보로는 알 수 없습니다"라고 답하세요.
2. 4M(Man·Machine·Material·Method) 분류 관점에서 원인을 설명하세요.
3. 답변은 현장 엔지니어가 실제로 참고할 수 있도록 구체적이고 간결하게 작성하세요.
4. 추측이나 일반 상식으로 답변을 지어내지 마세요.
"""

PARAMETER temperature 0.2
PARAMETER top_p 0.9
'''

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Modelfile 작성 완료")
print(modelfile_content)


Modelfile 작성 완료

FROM llama3.2:3b

SYSTEM """
당신은 자동차 엔진부품 제조 공장의 설비 진단 전문가입니다.
다음 원칙을 반드시 지켜 답변하세요.

1. 제공된 정보(context)에 근거해서만 답변하고, 모르면 "제공된 정보로는 알 수 없습니다"라고 답하세요.
2. 4M(Man·Machine·Material·Method) 분류 관점에서 원인을 설명하세요.
3. 답변은 현장 엔지니어가 실제로 참고할 수 있도록 구체적이고 간결하게 작성하세요.
4. 추측이나 일반 상식으로 답변을 지어내지 마세요.
"""

PARAMETER temperature 0.2
PARAMETER top_p 0.9



In [15]:
!ollama create engine-diagnosis-expert -f Modelfile


In [16]:
!ollama list


NAME                              ID              SIZE      MODIFIED               
engine-diagnosis-expert:latest    d41b1ccf5af8    2.0 GB    Less than a second ago    
qwen2.5:3b                        357c53fb659c    1.9 GB    2 minutes ago             
llama3.2:3b                       a80c4f17acd5    2.0 GB    3 minutes ago             
gemma:2b                          b50d6c999e59    1.7 GB    3 minutes ago             
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    3 minutes ago             


In [17]:
modelfile_content = modelfile_content.replace(
    "FROM llama3:8b-instruct-q4_0",
    "FROM llama3.2:3b"#"FROM gemma:2b"
)   #모델 바꿔서 검증

In [18]:
# 커스텀 모델 테스트 - 시스템 프롬프트만으로 역할이 고정되는지 확인
test_question = "치수불량이 발생했을 때 어떤 절차로 점검해야 하나요?"
answer = ask_ollama("engine-diagnosis-expert", test_question)
print(answer)


치수 불량이 발생한 경우, 4M(Man·Machine·Material·Method) 분류 관점에서 원인을 파악하기 위해 다음 절차를 따르세요.

1. **Machine(기계)**: 
   - 제조 공정 중에 발생한 치수 불량의 원인은 기계의 오류나 부적합한 기계 설정이 될 수 있습니다.
   - 이 경우, 기계의 상태와 설정을 확인하여 문제가 기계 자체에 대한지, 기계와 관련된 setting에 대한지 파악해야 합니다.

2. **Material(재료)**:
   - 치수 불량의 원인은 재료의 오류나 부적합한 재료가 될 수 있습니다.
   - 이 경우, 재료의 상태와 특성에 대해 확인하여 문제가 재료 자체에 대한지, 재료와 관련된 process에 대한지 파악해야 합니다.

3. **


## 4. 제조 QA 챗봇 프롬프트 설계: 원칙 적용 전후 비교

같은 질문에 대해 (a) 원칙 없는 단순 프롬프트, (b) 역할+제약+출력형식을 명시한 프롬프트로 응답 품질 비교


In [19]:
naive_prompt = "엔진 가공 공정에서 발생할 수 있는 불량 원인을 알려줘"

structured_prompt = """
당신은 자동차 엔진부품 제조 설비 진단 전문가입니다.
아래 형식으로만 답변하세요.

[질문]
엔진 가공 공정(CNC 가공, 정밀선반)에서 발생할 수 있는 불량 원인은 무엇인가?

[답변 형식]
1. 가능한 원인 (4M 분류 포함)
2. 일반적인 점검 방법
3. 확실하지 않은 부분은 "추가 확인이 필요합니다"라고 명시

답변은 3문장 이내로 간결하게 작성하세요.
"""

#모형 변경 검토: llama3:8b-instruct-q4_0
print("=== Naive 프롬프트 응답 ===")
print(ask_ollama("llama3.2:3b", naive_prompt))


print("\n\n=== 구조화 프롬프트 응답 ===")
print(ask_ollama("llama3.2:3b", structured_prompt))


=== Naive 프롬프트 응답 ===
엔진 가공 공정에서 발생할 수 있는 불량 원인은 다음과 같습니다.

1. **가공 중의 고온 및 고압**: 엔진 가공 공정은 높은 온도와 압력을 필요로 합니다. 그러나 이러한 조건이 too long or too high 인 경우, 엔진의 구조물이 파괴되거나 연기gas가 오염되는 등 불량을 일으킬 수 있습니다.
2. **가공 중의 물질 분해**: 가공 중의 물질 분해는 엔진의 재료가 파괴되고, 연기gas가 오염되는 등 불량을 일으킬 수 있습니다.
3. **가공 중의 열 방사**: 가공 중의 열 방사는 엔진의 구조물이 파괴되거나 연기gas가 오염되는 등 불량을 일으킬 수 있습니다.
4. **가공 중의 무시**: 가공 중의 무시


=== 구조화 프롬프트 응답 ===
[답변]

1. 가능한 원인은:
- 기계의 오류 (마진, 정밀도 등)
- 공정 조건의 불균형 (温度, 압력 등)
- 부품의 오류 (재질, shape 등)

2. 일반적인 점검 방법은:
- 엔진 가공 공정을 기록하고 분석하여 원인을 파악합니다.
- 기계와 공정 조건을ตรวจสอบ합니다.
- 부품을 확인하여 오류를 찾습니다.

3. 확실하지 않은 부분은 "추가 확인이 필요합니다"라고 명시.


## 5. LangChain + Ollama 연동: QA Chain 백엔드 교체

In [20]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-25 11:54:46--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861 (124K) [text/plain]
Saving to: ‘reports.json’

reports.json        100%[===================>] 123.89K  --.-KB/s    in 0.002s  

2026-06-25 11:54:46 (70.5 MB/s) - ‘reports.json’ saved [126861/126861]



In [21]:
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document          
from langchain_classic.chains import RetrievalQA       
import json

# 데이터 및 벡터스토어 재구성 (새 세션이라면 이 셀부터 실행)
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

#건수 제한(속도)
reports = reports[:150]

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
lc_documents = [
    Document(page_content=r["report_text"], metadata={"report_id": r["report_id"]})
    for r in reports
]
lc_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="lc_engine_reports_ollama",
)
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터스토어 재구성 완료")


/tmp/ipykernel_524/1742503557.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma as LC_Chroma
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

벡터스토어 재구성 완료


In [22]:
# LLM만 Ollama 커스텀 모델로 교체
ollama_llm = ChatOllama(model="engine-diagnosis-expert", temperature=0.2)

qa_chain_ollama = RetrievalQA.from_chain_type(
    llm=ollama_llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("Ollama 백엔드 QA Chain 구성 완료")


Ollama 백엔드 QA Chain 구성 완료


In [23]:
question = "포장 관련 불량 사례의 원인은 보통 무엇인가요?"
response = qa_chain_ollama.invoke({"query": question})

print("=== 답변 (Ollama 백엔드) ===")
print(response["result"])
print()
print("=== 참고한 출처 문서 ===")
for doc in response["source_documents"]:
    print("-", doc.metadata["report_id"], ":", doc.page_content[:80])


=== 답변 (Ollama 백엔드) ===
포장 관련 불량 사례의 원인은 보통 작업 표준이 미흡하거나, 부자재 투입 방향성이 부족하여 포장 오류가 발생하는 경우가 많습니다.

=== 참고한 출처 문서 ===
- RPT-078 : 2024년 8월 23일 15:45, 자동포장기 B-1호기에서 포장 후 동봉되는 플라스틱 캡과 설명서 방향이 서로 반대로 삽입된 제품을 확인함. 
- RPT-033 : 2024년 4월 7일 10:10, 컨베이어 벨트 3라인에서 플라스틱 커버 이송 후 외관면에 길이 15~28mm의 선형 스크래치가 발생한 것을 확
- RPT-076 : 2024년 8월 17일 09:15, 자동포장기 A-1호기에서 포장 파우치 절단 길이가 기준 180.0mm 대비 176.8~177.5mm로 짧게 


# 6. 민원 데이터에 적용

- 카드사 VOC

In [24]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/voc_card.zip

--2026-06-25 11:55:45--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/voc_card.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1068642 (1.0M) [application/zip]
Saving to: ‘voc_card.zip’

voc_card.zip        100%[===================>]   1.02M  --.-KB/s    in 0.007s  

2026-06-25 11:55:45 (154 MB/s) - ‘voc_card.zip’ saved [1068642/1068642]



In [25]:
!unzip -O CP949 voc_card.zip -d ./voc_card/

Archive:  voc_card.zip
  inflating: ./voc_card/하나카드_1537.json  
  inflating: ./voc_card/하나카드_1540.json  
  inflating: ./voc_card/하나카드_1532.json  
  inflating: ./voc_card/하나카드_1548.json  
  inflating: ./voc_card/하나카드_1563.json  
  inflating: ./voc_card/하나카드_1538.json  
  inflating: ./voc_card/하나카드_1542.json  
  inflating: ./voc_card/하나카드_1535.json  
  inflating: ./voc_card/하나카드_1541.json  
  inflating: ./voc_card/하나카드_1536.json  
  inflating: ./voc_card/하나카드_1555.json  
  inflating: ./voc_card/하나카드_1544.json  
  inflating: ./voc_card/하나카드_1580.json  
  inflating: ./voc_card/하나카드_1545.json  
  inflating: ./voc_card/하나카드_1534.json  
  inflating: ./voc_card/하나카드_1533.json  
  inflating: ./voc_card/하나카드_1539.json  
  inflating: ./voc_card/하나카드_1561.json  
  inflating: ./voc_card/하나카드_1575.json  
  inflating: ./voc_card/하나카드_1554.json  
  inflating: ./voc_card/하나카드_1573.json  
  inflating: ./voc_card/하나카드_1582.json  
  inflating: ./voc_card/하나카드_1550.json  
  inflating: ./voc_card/하나카드_1547.

In [26]:
import json
import glob

# 모든 json 파일 읽기
json_files = glob.glob('./voc_card/*.json')

data_list = []
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        data_list.append(data[0])

print(f"총 {len(data_list)}개 파일 로드")
print(data_list[0])  # 첫 번째 확인

총 776개 파일 로드
{'source_id': '23861', 'source': '하나카드', 'consulting_date': '', 'consulting_category': '선결제/즉시출금', 'client_gender': '여자', 'client_age': '30대', 'consulting_turns': '36', 'consulting_length': 253, 'consulting_content': '상담사: 상담원 ▲▲▲입니다.\n손님: 아 네, 제가 선결제를 할려고 하니까요\n상담사: 네.\n손님: 이게 지금 선결제 가 안 되는데요.\n상담사: 아 그러세요? 한번 확인해보겠습니다.\n손님: 예.\n상담사: 네, 명의자분 ▲▲▲ 님 본인 맞으실까요?\n손님: 네.\n상담사: 네, 확인 감사합니다. 어플에서 진행이 안 되시는 걸까요?\n손님: 네, 어플에서 진행할려고 하니까 오래 쓰면서 금감원 개인 정보 노출자 사고 예방 시스템 등록계좌라고 뜨거든요.\n상담사: 네.\n상담사: 음 제가 한번 출금 시도 해볼 텐데 ▲▲에서 이번 달 카드 대금 하시는거 맞으시죠?\n손님: 네.\n상담사: 네 잠시만 기다려 주세요.\n손님: 네.\n상담사: 기다려주셔서 감사합니다. 정상적으로 출금은 완료해드렸고요.\n손님: 네.\n상담사: 혹시 이거 저희 쪽에도 ▲▲년도 ▲▲월 쯤에 카드 발급 방지 등재 하셨던거 있는데 이때 혹시 같이 등록하셨던 건 없으실까요?\n손님: 네.\n손님: 모르겠어요. 기억이 안나요.\n상담사: ▲▲월 달쯤에 아예 저희 금융감독원 쪽에다가\n손님: 네.\n상담사: 발급 방지라고 해서 이게 손님 명의의 카드가 더 이상 발급되지 않도록 등록해 놓으시는 게 있어요. 그게 일단 지금 등록이 되어 있고 아마 이거 하시면서 은행쪽에도 뭔가 등록을 하셔야지 않으실까 싶습니다.\n손님: 예.\n손님: 어 그럼 이거를 해지할려면 금감원에 전화해야 돼요?\n상담사: 아니 금감원 사이트에 직접 접속하실 수 있고 \n손님: 네.\n상담사: 이거 저희가 카드 관련된 

- QA Chain

In [27]:
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document          
from langchain_classic.chains import RetrievalQA       
import json

In [28]:
# 데이터 및 벡터스토어 재구성 (새 세션이라면 이 셀부터 실행)
reports = data_list[:150]

In [29]:

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
lc_documents = [
    Document(page_content=r["consulting_content"], metadata={"report_id": r["source_id"]})
    for r in reports
]
lc_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="lc_voc_card_ollama",
)
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터스토어 재구성 완료")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터스토어 재구성 완료


In [30]:
# LLM만 Ollama 커스텀 모델
ollama_llm = ChatOllama(model="engine-diagnosis-expert", temperature=0.2)

qa_chain_ollama = RetrievalQA.from_chain_type(
    llm=ollama_llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("Ollama 백엔드 QA Chain 구성 완료")


Ollama 백엔드 QA Chain 구성 완료


In [31]:
question = "카드 요금 납부 시 문제는 무엇인가요?"
response = qa_chain_ollama.invoke({"query": question})

print("=== 답변 (Ollama 백엔드) ===")
print(response["result"])
print()
print("=== 참고한 출처 문서 ===")
for doc in response["source_documents"]:
    print("-", doc.metadata["report_id"], ":", doc.page_content[:80])


=== 답변 (Ollama 백엔드) ===
카드 요금 납부 시的问题 중 하나가 "문자 알림이 없기 때문에"라는 경우입니다. 카드 사용 내역을 확인하기 위해 문자를 보내야 하는데, 이때문자 알림이 없다는 것이 문제입니다.

=== 참고한 출처 문서 ===
- 23870 : 상담사: 상담원 ▲▲▲입니다.
손님: 아 네, 문의 좀 드릴려고 하는데요. 카드 그 뭐지 사용 기한이
상담사: 네.
손님: 마무리가 돼 가지고 
- 23821 : 상담사: 상담원 ▲▲▲입니다.
손님: 안녕하세요? 
상담사: 네, 안녕하세요? 
손님: 다름이 아니고 아니 전기 요금을 제가 카드로 자동이체 해
- 21596 : 상담사: 상담원 ▲▲▲입니다. 무엇을 도와드릴까요?
손님: 네, 안녕하세요? 그 카드 사용 중인데.
상담사: 네.
손님: 그 사용 기간을 변경하


# 7. streamlit+rag

- 로컬에서 설치 및 실행

In [ ]:
!pip install streamlit langchain-ollama langchain-core langchain-community \
    langchain-text-splitters langchain-huggingface sentence-transformers \
    pymupdf pdfplumber chromadb torchvision \
    torch --index-url https://download.pytorch.org/whl/cpu      

- ollama 설치: https://ollama.com/download/
- 설치 후, 터미널에서 ollama --version, ollama list 확인
- 이후 conda activate 환경명
- ollama pull qwen2.5:3b

- app2_1.py 실행

In [ ]:
import streamlit as st
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

st.title("엔진 진단 챗봇")

# 모델 초기화 (세션당 1회)
if "llm" not in st.session_state:
    st.session_state.llm = ChatOllama(
        model="qwen2.5:3b",
        temperature=0.2,
        timeout=120,
    )

# 대화 기록 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 출력
for msg in st.session_state.messages:
    role = "user" if isinstance(msg, HumanMessage) else "assistant"
    with st.chat_message(role):
        st.write(msg.content)

# 입력
#prompt = st.chat_input("질문")
#if prompt:
#    ...
if prompt := st.chat_input("질문을 입력하세요"):        #월루스 연산자, 할당 표현식
    # 사용자 메시지 추가 및 출력
    st.session_state.messages.append(HumanMessage(content=prompt))
    with st.chat_message("user"):
        st.write(prompt)

    # LLM 호출 (전체 대화 기록 전달 → 멀티턴)
    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            response = st.session_state.llm.invoke(st.session_state.messages)
        st.write(response.content)

    # 응답 저장
    st.session_state.messages.append(AIMessage(content=response.content))

- app2_2.py

In [ ]:
import streamlit as st
from langchain_ollama import ChatOllama
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

st.title("엔진 진단 챗봇 (LangChain 메모리)")

# 세션별 메모리 저장소
if "store" not in st.session_state:
    st.session_state.store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in st.session_state.store:
        st.session_state.store[session_id] = InMemoryChatMessageHistory()
    return st.session_state.store[session_id]

# 체인 초기화
if "chain" not in st.session_state:
    llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, timeout=120)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 제조업 엔진 진단 전문가입니다."),
        MessagesPlaceholder(variable_name="history"),  # 대화 기록 자동 삽입
        ("human", "{input}"),
    ])

    chain = prompt | llm

    st.session_state.chain = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

# 대화 출력
for msg in get_session_history("user1").messages:
    role = "user" if msg.type == "human" else "assistant"
    with st.chat_message(role):
        st.write(msg.content)

# 입력
if prompt := st.chat_input("질문을 입력하세요"):
    with st.chat_message("user"):
        st.write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            response = st.session_state.chain.invoke(
                {"input": prompt},
                config={"configurable": {"session_id": "user1"}}
            )
        st.write(response.content)